# Fill missing focus scores with respondent preference models

This notebook trains one model per `respondentId` from `data/raw/4_comfort_perception.csv`, joined to nearby room measurements, then randomly selects among those respondent models to fill `focus_score` for every row in the large unified environmental dataset. The filled target is on the 1-5 comfort/focus scale.

In [1]:
from pathlib import Path
import sys

import pandas as pd

MAL_DIR = Path.cwd()
if MAL_DIR.name != "MAL":
    MAL_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "MAL")

sys.path.insert(0, str(MAL_DIR))

from scripts.build_unified_environment_dataset import build_unified_dataset
from scripts.fill_missing_targets import (
    build_respondent_comfort_training_data,
    fill_focus_scores_from_respondent_preferences,
)

PROCESSED_DIR = MAL_DIR / "data" / "processed"
UNIFIED_PATH = PROCESSED_DIR / "unified_environment_focus_dataset.csv"
REPORT_PATH = PROCESSED_DIR / "unified_environment_focus_dataset_report.csv"
ROOM_MEASUREMENTS_PATH = MAL_DIR / "data" / "raw" / "1_room_measurements.csv"
COMFORT_PERCEPTION_PATH = MAL_DIR / "data" / "raw" / "4_comfort_perception.csv"
RESPONDENT_TRAINING_PATH = PROCESSED_DIR / "respondent_comfort_training_dataset.csv"
FILLED_FOCUS_PATH = PROCESSED_DIR / "unified_environment_respondent_focus_score_filled.csv"

UNIFIED_PATH, COMFORT_PERCEPTION_PATH, FILLED_FOCUS_PATH

(PosixPath('/home/edf/Documents/GitSynced/SEP4_/MAL/data/processed/unified_environment_focus_dataset.csv'), PosixPath('/home/edf/Documents/GitSynced/SEP4_/MAL/data/raw/4_comfort_perception.csv'), PosixPath('/home/edf/Documents/GitSynced/SEP4_/MAL/data/processed/unified_environment_respondent_focus_score_filled.csv'))

## Load environmental rows and respondent comfort data

In [2]:
if not UNIFIED_PATH.exists():
    unified_df, source_report = build_unified_dataset(
        output_path=UNIFIED_PATH,
        report_path=REPORT_PATH,
        allow_missing=False,
    )
else:
    unified_df = pd.read_csv(UNIFIED_PATH, low_memory=False)

measurements_df = pd.read_csv(ROOM_MEASUREMENTS_PATH, low_memory=False)
comfort_df = pd.read_csv(COMFORT_PERCEPTION_PATH, low_memory=False)

input_summary = pd.DataFrame([
    {
        "environment_rows_to_fill": len(unified_df),
        "comfort_response_rows": len(comfort_df),
        "respondent_models_available": int(comfort_df["respondentId"].nunique()),
        "comfort_value_classes": sorted(comfort_df["comfortValue"].dropna().astype(int).unique().tolist()),
    }
])
input_summary

environment_rows_to_fill,comfort_response_rows,respondent_models_available,comfort_value_classes
866348,1830,182,"[1, 2, 3, 4, 5]"


## Build respondent training rows

In [3]:
respondent_training_df = build_respondent_comfort_training_data(measurements_df, comfort_df)
respondent_training_df.to_csv(RESPONDENT_TRAINING_PATH, index=False)

respondent_training_summary = pd.DataFrame([
    {
        "training_rows": len(respondent_training_df),
        "respondents": int(respondent_training_df["respondent_id"].nunique()),
        "missing_focus_score": int(respondent_training_df["focus_score"].isna().sum()),
    }
])
respondent_training_summary

training_rows,respondents,missing_focus_score
1830,182,0


In [4]:
respondent_training_df.groupby("respondent_id")["focus_score"].agg(
    training_rows="size",
    score_classes="nunique",
    mean_score="mean",
).describe()

,training_rows,score_classes,mean_score
count,182.000000,182.000000,182.000000
mean,10.054945,2.445055,3.361490
std,6.525534,1.153784,0.584953
min,1.000000,1.000000,2.000000
25%,3.000000,1.000000,3.000000
50%,10.500000,2.000000,3.250000
75%,16.000000,3.000000,3.774510
max,21.000000,5.000000,5.000000


## Train one model per respondent and fill 1-5 focus_score

The shared environmental features are `humidity`, `temperature`, `noise`, and `co2`. Rows that already had `focus_score` keep their original value; missing rows are filled by randomly selecting one of the respondent models.

In [5]:
filled_df, model_summary = fill_focus_scores_from_respondent_preferences(
    unified_df,
    respondent_training_df,
    feature_columns=["humidity", "temperature", "noise", "co2"],
    random_state=42,
)

filled_df.to_csv(FILLED_FOCUS_PATH, index=False)
model_summary

student_id,training_rows,classes
1,1,4
2,9,"2,3,4,5"
3,8,"3,4"
4,19,"2,3,4,5"
5,12,"2,3"
6,1,3
7,14,"2,3,4,5"
8,20,"3,4,5"
9,10,"2,3"
10,14,"3,5"


## Verify every row has a 1-5 target

In [6]:
target_check = pd.DataFrame([
    {
        "output_csv": str(FILLED_FOCUS_PATH),
        "rows": len(filled_df),
        "rows_with_focus_score": int(filled_df["focus_score"].notna().sum()),
        "rows_missing_focus_score": int(filled_df["focus_score"].isna().sum()),
        "all_rows_have_focus_score": bool(filled_df["focus_score"].notna().all()),
        "respondent_models_trained": len(model_summary),
        "min_focus_score": int(filled_df["focus_score"].min()),
        "max_focus_score": int(filled_df["focus_score"].max()),
    }
])
target_check

output_csv,rows,rows_with_focus_score,rows_missing_focus_score,all_rows_have_focus_score,respondent_models_trained,min_focus_score,max_focus_score
/home/edf/Documents/GitSynced/SEP4_/MAL/data/processed/unified_environment_respondent_focus_score_filled.csv,866348,866348,0,True,182,1,5


In [7]:
assert filled_df["focus_score"].notna().all(), "Some rows still have missing focus_score"
assert filled_df["focus_score"].between(1, 5).all(), "focus_score should be on the 1-5 scale"
print("All rows have 1-5 focus_score values filled by randomly selected respondent models.")

All rows have 1-5 focus_score values filled by randomly selected respondent models.


## Focus score distribution

In [8]:
focus_score_distribution = (
    filled_df["focus_score"]
    .value_counts()
    .sort_index()
    .rename_axis("focus_score")
    .reset_index(name="rows")
)
focus_score_distribution

focus_score,rows
1,8810
2,93781
3,424761
4,251384
5,87612


In [9]:
pd.crosstab(filled_df["source"], filled_df["focus_score"])

focus_score,1,2,3,4,5
source,,,,,
homecoach_5min_2023,1024,8200,38784,21672,7365
homecoach_5min_2024,1066,9993,51241,28414,9611
homecoach_5min_2025,1268,10103,51134,28126,9559
homecoach_5min_2026,268,2985,18231,9422,3226
keti_1min_resampled,5143,62070,263096,162464,57388
room_conditions,13,251,1386,777,238
room_measurements_with_comfort,28,179,889,509,225


## Preview of the saved filled dataset

In [10]:
filled_df.head(20)

timestamp,session_id,location_id,record_id,source,humidity,light,temperature,noise,co2,focus_score
2013-08-23T16:06:00Z,keti_1min_resampled__510__s00001,510,keti_1min_resampled:row_00192329,keti_1min_resampled,NaN,NaN,NaN,NaN,316.000000,4
2013-08-23T16:24:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00011995,keti_1min_resampled,NaN,NaN,NaN,NaN,476.555556,2
2013-08-23T16:25:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00011996,keti_1min_resampled,NaN,NaN,NaN,NaN,480.333333,5
2013-08-23T16:26:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00011997,keti_1min_resampled,NaN,NaN,NaN,NaN,485.166667,3
2013-08-23T16:27:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00011998,keti_1min_resampled,NaN,NaN,NaN,NaN,481.083333,3
2013-08-23T16:28:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00011999,keti_1min_resampled,NaN,NaN,NaN,NaN,482.250000,3
2013-08-23T16:29:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00012000,keti_1min_resampled,NaN,NaN,NaN,NaN,483.333333,4
2013-08-23T16:30:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00012001,keti_1min_resampled,NaN,NaN,NaN,NaN,480.000000,3
2013-08-23T16:31:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00012002,keti_1min_resampled,NaN,NaN,NaN,NaN,475.833333,4
2013-08-23T16:32:00Z,keti_1min_resampled__415__s00001,415,keti_1min_resampled:row_00012003,keti_1min_resampled,NaN,NaN,NaN,NaN,488.833333,3
